# Notebook 06: modelado y evaluación

**RSNA 2022 Cervical Spine Fracture Detection — Proyecto 2, fase de resultados**

Entrena y compara tres clasificadores binarios a nivel de **corte axial** y exporta los
artefactos que consume la aplicación Streamlit de `Aplicacion/`.

## Definición de la tarea

El conjunto etiqueta **estudios**, no cortes. Para obtener una tarea 2D supervisada y
honesta se usa la anotación espacial:

| Clase | Origen | Criterio |
|---|---|---|
| Positivo | 7 217 cortes con bounding box | la lesión es visible en ese corte |
| Negativo fácil | estudios con `patient_overall = 0` | ninguna vértebra fracturada |
| Negativo difícil | estudios positivos, cortes a >20 de toda caja | hueso sano del mismo paciente; se toma la mitad de los cortes anotados del estudio |

Los negativos difíciles son necesarios: sin ellos el modelo puede separar las clases
reconociendo al *paciente* o su protocolo de adquisición en lugar de la discontinuidad
ósea. El EDA ya mostró una asociación pequeña pero real entre metadatos de adquisición y
etiqueta (`n_slices`, p = 1.8e-8), y este es el mecanismo que la neutraliza.

**Los estudios positivos sin bounding box se excluyen por completo**: se sabe que hay
fractura pero no en qué corte, así que ningún corte suyo tiene etiqueta fiable.

## Límite de interpretación

Un modelo por corte **no diagnostica al paciente**. Responde "¿se ve una fractura en esta
imagen?", no "¿este paciente tiene una fractura cervical?". Agregar cortes a un veredicto
por estudio es un problema distinto que este notebook no resuelve.

In [ ]:
# Muchos DICOM del conjunto usan JPEG Lossless. Sin decodificador, pydicom falla
# al leer los pixeles y los cortes se pierden en silencio.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pylibjpeg>=2.0", "pylibjpeg-libjpeg>=2.1", "python-gdcm"], check=False)
print("decodificadores instalados")

In [ ]:
import json, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA)

BASE = next((p for p in [
    Path("/kaggle/input/rsna-2022-cervical-spine-fracture-detection"),
    Path("/kaggle/input/competitions/rsna-2022-cervical-spine-fracture-detection"),
] if (p / "train.csv").is_file()), None)
if BASE is None:
    cand = [p.parent for p in Path("/kaggle/input").glob("*/train.csv")]
    BASE = cand[0] if cand else None
if BASE is None:
    raise FileNotFoundError(f"dataset no montado. Inputs: {[p.name for p in Path('/kaggle/input').iterdir()]}")

IMAGES = BASE / "train_images"
SALIDA = Path("/kaggle/working")
(SALIDA / "models").mkdir(exist_ok=True)
(SALIDA / "metrics").mkdir(exist_ok=True)
(SALIDA / "figures").mkdir(exist_ok=True)

train = pd.read_csv(BASE / "train.csv")
bbox = pd.read_csv(BASE / "train_bounding_boxes.csv")
bbox["StudyInstanceUID"] = bbox["StudyInstanceUID"].astype(str)
print(f"Dataset   : {BASE}")
print(f"Estudios  : {len(train):,} | positivos: {int(train.patient_overall.sum()):,}")
print(f"Cajas     : {len(bbox):,} sobre {bbox.StudyInstanceUID.nunique()} estudios")
import tensorflow as tf
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Construcción del inventario de cortes

In [ ]:
uids_pos = set(train.loc[train.patient_overall.eq(1), "StudyInstanceUID"].astype(str))
uids_neg = set(train.loc[train.patient_overall.eq(0), "StudyInstanceUID"].astype(str))
uids_bbox = set(bbox.StudyInstanceUID)

# Cortes positivos: uno por (estudio, corte) anotado.
positivos = (bbox.groupby(["StudyInstanceUID", "slice_number"]).size()
             .reset_index(name="n_cajas")
             .rename(columns={"slice_number": "corte"}))
positivos["etiqueta"] = 1
positivos["tipo"] = "positivo"
print(f"Cortes positivos: {len(positivos):,} en {positivos.StudyInstanceUID.nunique()} estudios")

from functools import lru_cache

@lru_cache(maxsize=None)
def cortes_de(uid):
    d = IMAGES / uid
    return tuple(sorted(int(f.stem) for f in d.glob("*.dcm") if f.stem.isdigit())) if d.is_dir() else ()

MARGEN = 20  # cortes de distancia minima a cualquier caja

# Negativos dificiles: hueso sano del mismo paciente positivo.
anotados = bbox.groupby("StudyInstanceUID")["slice_number"].apply(set).to_dict()
filas = []
rng = np.random.default_rng(SEMILLA)
for uid in sorted(uids_bbox):
    todos = cortes_de(uid)
    if not todos:
        continue
    marcados = anotados[uid]
    lejanos = [c for c in todos if min(abs(c - a) for a in marcados) > MARGEN]
    # Solo la MITAD de los negativos sale del propio paciente: si se toman tantos
    # como positivos, los negativos faciles se quedan sin cupo y el conjunto se
    # reduce a los 235 estudios anotados.
    k = min(len(marcados) // 2, len(lejanos))
    for c in rng.choice(lejanos, size=k, replace=False) if k else []:
        filas.append({"StudyInstanceUID": uid, "corte": int(c), "etiqueta": 0, "tipo": "negativo_dificil"})
neg_dificiles = pd.DataFrame(filas)
print(f"Negativos dificiles: {len(neg_dificiles):,}")

In [ ]:
# Negativos faciles: estudios sin fractura, muestreados en las MISMAS posiciones
# relativas que ocupan los positivos, para que la posicion en la pila no sea una pista.
pos_rel = []
for uid, grupo in positivos.groupby("StudyInstanceUID"):
    todos = cortes_de(uid)
    if todos:
        lo, hi = min(todos), max(todos)
        span = max(hi - lo, 1)
        pos_rel.extend(((grupo["corte"] - lo) / span).tolist())
pos_rel = np.clip(np.array(pos_rel), 0, 1)
print(f"Posicion relativa de los positivos: mediana {np.median(pos_rel):.3f}")

objetivo = len(positivos) - len(neg_dificiles)
objetivo = max(objetivo, 0)
negs_elegibles = sorted(uids_neg)
rng.shuffle(negs_elegibles)

filas = []
por_estudio = max(1, int(np.ceil(objetivo / max(len(negs_elegibles), 1))))
for uid in negs_elegibles:
    if len(filas) >= objetivo:
        break
    todos = cortes_de(uid)
    if not todos:
        continue
    lo, hi = min(todos), max(todos)
    span = max(hi - lo, 1)
    for r in rng.choice(pos_rel, size=min(por_estudio, len(todos)), replace=True):
        c = int(round(lo + r * span))
        filas.append({"StudyInstanceUID": uid, "corte": c, "etiqueta": 0, "tipo": "negativo_facil"})
neg_faciles = pd.DataFrame(filas).drop_duplicates(["StudyInstanceUID", "corte"]).head(objetivo)
print(f"Negativos faciles: {len(neg_faciles):,}")

inventario = pd.concat([
    positivos[["StudyInstanceUID", "corte", "etiqueta", "tipo"]],
    neg_dificiles, neg_faciles,
], ignore_index=True).drop_duplicates(["StudyInstanceUID", "corte"]).reset_index(drop=True)

print(f"\nINVENTARIO: {len(inventario):,} cortes")
print(inventario.tipo.value_counts().to_string())
print(f"Balance: {100*inventario.etiqueta.mean():.1f}% positivos")
inventario.to_csv(SALIDA / "inventario_cortes.csv", index=False)

## 2. Partición agrupada por paciente

La partición agrupa por `StudyInstanceUID`: ningún corte de un mismo paciente puede caer
a la vez en entrenamiento y prueba. Es la conclusión directa del EDA — sin agrupar, cortes
contiguos casi idénticos quedarían a ambos lados y la métrica saldría inflada.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

grupos = inventario["StudyInstanceUID"].values
idx_tr, idx_tmp = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEMILLA)
                       .split(inventario, inventario.etiqueta, grupos))
tmp = inventario.iloc[idx_tmp]
idx_va, idx_te = next(GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEMILLA)
                      .split(tmp, tmp.etiqueta, tmp["StudyInstanceUID"].values))

part_tr = inventario.iloc[idx_tr].reset_index(drop=True)
part_va = tmp.iloc[idx_va].reset_index(drop=True)
part_te = tmp.iloc[idx_te].reset_index(drop=True)

for nombre, p in [("train", part_tr), ("val", part_va), ("test", part_te)]:
    print(f"{nombre:6s}: {len(p):6,} cortes | {p.StudyInstanceUID.nunique():4d} estudios | "
          f"{100*p.etiqueta.mean():5.1f}% positivos")

fuga = (set(part_tr.StudyInstanceUID) & set(part_te.StudyInstanceUID)) | \
       (set(part_tr.StudyInstanceUID) & set(part_va.StudyInstanceUID))
assert not fuga, f"FUGA DE PACIENTES: {len(fuga)} estudios compartidos"
print("\nSin fuga de pacientes entre particiones.")

## 3. Lectura de píxeles: HU y ventana ósea

Cada corte se convierte a unidades Hounsfield con `RescaleSlope`/`RescaleIntercept` y se
le aplica la ventana ósea (centro 400, ancho 1 800) que el notebook 04 mostró necesaria.
Sin ella el hueso trabecular pierde el contraste donde se manifiesta la fractura.

In [ ]:
import pydicom
from concurrent.futures import ThreadPoolExecutor

LADO = 224
CENTRO, ANCHO = 400.0, 1800.0

def carga_corte(args):
    uid, corte = args
    try:
        ds = pydicom.dcmread(IMAGES / uid / f"{corte}.dcm")
        hu = ds.pixel_array.astype(np.float32)
        hu = hu * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
        v = np.clip((hu - (CENTRO - ANCHO / 2)) / ANCHO, 0, 1)
        img = tf.image.resize(v[..., None], (LADO, LADO), method="bilinear").numpy()
        return (np.clip(img * 255, 0, 255).astype(np.uint8), True)
    except Exception:
        return (np.zeros((LADO, LADO, 1), np.uint8), False)

def construir(parte, nombre):
    t0 = time.time()
    pares = list(zip(parte.StudyInstanceUID, parte.corte))
    with ThreadPoolExecutor(max_workers=8) as ex:
        res = list(ex.map(carga_corte, pares))
    X = np.stack([r[0] for r in res])
    ok = np.array([r[1] for r in res])
    y = parte.etiqueta.values.astype(np.float32)
    fallos = int((~ok).sum())
    tasa = fallos / max(len(ok), 1)
    print(f"{nombre:6s}: {X[ok].shape} | fallos: {fallos} ({100*tasa:.1f}%) | {time.time()-t0:.0f}s")
    if tasa > 0.05:
        raise RuntimeError(
            f"{nombre}: {100*tasa:.1f}% de los cortes no se pudieron leer. "
            "Casi siempre falta el decodificador JPEG Lossless (pylibjpeg)."
        )
    return X[ok], y[ok], parte[ok].reset_index(drop=True)

X_tr, y_tr, part_tr = construir(part_tr, "train")
X_va, y_va, part_va = construir(part_va, "val")
X_te, y_te, part_te = construir(part_te, "test")
print(f"\nMemoria: {(X_tr.nbytes + X_va.nbytes + X_te.nbytes)/1e9:.2f} GB")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for fila, (etiqueta, titulo) in enumerate([(1, "Positivo"), (0, "Negativo")]):
    idx = np.where(y_tr == etiqueta)[0][:6]
    for col, i in enumerate(idx):
        axes[fila, col].imshow(X_tr[i, :, :, 0], cmap="gray", vmin=0, vmax=255)
        axes[fila, col].set_title(f"{titulo} · {part_tr.tipo.iloc[i][:14]}", fontsize=8)
        axes[fila, col].axis("off")
fig.suptitle("Muestras de entrenamiento en ventana ósea (224x224)", fontweight="bold")
fig.tight_layout(); fig.savefig(SALIDA / "figures/06_muestras_entrenamiento.png", dpi=200, bbox_inches="tight")
plt.close(fig); print("figura guardada")

## 4. Los tres modelos

Se entrenan tres arquitecturas de capacidad creciente para que la comparación diga algo:
si la transferencia no supera a la CNN pequeña, el cuello de botella son los datos y no
la capacidad del modelo.

| Modelo | Parámetros | Razón |
|---|---|---|
| `cnn_simple` | ~0.3 M, desde cero | línea base honesta; sin ella no se sabe cuánto aporta la transferencia |
| `mobilenetv2` | 2.3 M, ImageNet | transferencia ligera, la más rápida en inferencia para la app |
| `efficientnetb0` | 4.0 M, ImageNet | mayor capacidad; comprueba si más parámetros ayudan con este volumen |

Las dos redes preentrenadas esperan 3 canales, así que el corte en gris se replica. La app
recibe esa configuración en `models.yaml` (`color_mode: rgb`, `channels: 3`) y reproduce
exactamente el mismo preprocesamiento.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def aumento():
    # Se construye uno nuevo por modelo: reutilizar capas tras clear_session()
    # las deja apuntando a un grafo destruido.
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.04),
        layers.RandomZoom(0.08),
        layers.RandomTranslation(0.05, 0.05),
    ], name="aumento")

# El modelo recibe pixeles CRUDOS en [0,255] y normaliza internamente. Asi
# models.yaml declara `normalization: none` y la app no puede normalizar dos
# veces: el preprocesamiento vive en un solo sitio, dentro del artefacto.
def cnn_simple(canales=1):
    e = keras.Input((LADO, LADO, canales))
    x = aumento()(e)
    x = layers.Rescaling(1 / 255.0)(x)
    for filtros in (32, 64, 128, 128):
        x = layers.Conv2D(filtros, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    s = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(e, s, name="cnn_simple"), None

def transferencia(constructor, preparar, nombre):
    # `preparar` es una CAPA serializable, no una funcion envuelta en Lambda: un
    # Lambda que apunta a preprocess_input no se puede volver a cargar desde el
    # .keras sin pasar custom_objects, y la app fallaria al abrir el modelo.
    e = keras.Input((LADO, LADO, 3))
    x = aumento()(e)
    if preparar is not None:
        x = preparar(x)
    base = constructor(include_top=False, weights="imagenet",
                       input_shape=(LADO, LADO, 3), pooling="avg")
    base.trainable = False                      # etapa 1: solo la cabeza
    x = base(x, training=False)
    x = layers.Dropout(0.3)(x)
    s = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(e, s, name=nombre), base

def a_rgb(X):
    return np.repeat(X, 3, axis=-1)

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)

EPOCAS_1, EPOCAS_2, LOTE = 8, 6, 64

def entrenar(modelo, base, Xtr, Xva, afinar=False):
    t0 = time.time()
    paradas = [keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3,
                                             restore_best_weights=True, verbose=0)]
    modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss="binary_crossentropy",
                   metrics=[keras.metrics.AUC(name="auc")])
    h1 = modelo.fit(Xtr, y_tr, validation_data=(Xva, y_va), epochs=EPOCAS_1,
                    batch_size=LOTE, callbacks=paradas, verbose=2)
    historia = {k: list(map(float, v)) for k, v in h1.history.items()}
    if afinar and base is not None:
        # Etapa 2: se descongela la base con un learning rate 100x menor.
        base.trainable = True
        modelo.compile(optimizer=keras.optimizers.Adam(1e-5),
                       loss="binary_crossentropy",
                       metrics=[keras.metrics.AUC(name="auc")])
        h2 = modelo.fit(Xtr, y_tr, validation_data=(Xva, y_va), epochs=EPOCAS_2,
                        batch_size=LOTE, callbacks=paradas, verbose=2)
        for k, v in h2.history.items():
            historia.setdefault(k, []).extend(map(float, v))
    return modelo, historia, time.time() - t0

def umbral_youden(modelo, Xva):
    """Elige el umbral que maximiza sensibilidad + especificidad - 1 en VALIDACION.

    Usar 0.5 por defecto supone que el modelo esta calibrado, y no tiene por que
    estarlo: una red puede ordenar bien (AUC alto) y aun asi colocar casi todas
    sus puntuaciones a un lado de 0.5. El umbral se elige en validacion y se
    aplica sin tocar a prueba.
    """
    p = modelo.predict(Xva, batch_size=128, verbose=0).ravel()
    fpr, tpr, umbrales = roc_curve(y_va, p)
    j = tpr - fpr
    return float(umbrales[int(np.argmax(j))])

def evaluar(modelo, Xte, nombre, umbral=0.5):
    t0 = time.time()
    p = modelo.predict(Xte, batch_size=128, verbose=0).ravel()
    ms_por_imagen = 1000 * (time.time() - t0) / len(Xte)
    yhat = (p >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, yhat, labels=[0, 1]).ravel()
    return {
        "modelo": nombre,
        "accuracy": float(accuracy_score(y_te, yhat)),
        "precision": float(precision_score(y_te, yhat, zero_division=0)),
        "recall": float(recall_score(y_te, yhat, zero_division=0)),
        "sensitivity": float(recall_score(y_te, yhat, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) else 0.0,
        "f1": float(f1_score(y_te, yhat, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_te, p)),
        "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
        "umbral": float(umbral),
        "ms_por_imagen": float(ms_por_imagen),
        "n_test": int(len(y_te)),
    }, p

In [ ]:
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
# Equivalentes serializables de los preprocess_input oficiales, verificados
# numericamente: efficientnet.preprocess_input es identidad (la red lleva su
# propia normalizacion dentro) y mobilenet_v2.preprocess_input es x/127.5 - 1.
prep_eff = None
prep_mob = layers.Rescaling(1 / 127.5, offset=-1.0, name="preprocess_mobilenet")

resultados, curvas, historias = [], {}, {}

ESPECS = [
    ("cnn_simple",     "CNN propia (desde cero)",      lambda: cnn_simple(1), False, 1),
    ("mobilenetv2",    "MobileNetV2 (ImageNet)",       lambda: transferencia(MobileNetV2, prep_mob, "mobilenetv2"), True, 3),
    ("efficientnetb0", "EfficientNetB0 (ImageNet)",    lambda: transferencia(EfficientNetB0, prep_eff, "efficientnetb0"), True, 3),
]

for ident, titulo, construir_modelo, afinar, canales in ESPECS:
    print(f"\n{'='*64}\n{titulo}\n{'='*64}")
    Xtr = a_rgb(X_tr) if canales == 3 else X_tr
    Xva = a_rgb(X_va) if canales == 3 else X_va
    Xte = a_rgb(X_te) if canales == 3 else X_te
    keras.backend.clear_session()
    modelo, base = construir_modelo()
    modelo, hist, segundos = entrenar(modelo, base, Xtr, Xva, afinar=afinar)
    umbral = umbral_youden(modelo, Xva)
    met, p = evaluar(modelo, Xte, titulo, umbral=umbral)
    met_05, _ = evaluar(modelo, Xte, titulo, umbral=0.5)
    met["sensitivity_umbral_05"] = met_05["sensitivity"]
    met["specificity_umbral_05"] = met_05["specificity"]
    met["f1_umbral_05"] = met_05["f1"]
    print(f"  umbral elegido en validacion: {umbral:.4f} "
          f"(con 0.5 daria sens {met_05['sensitivity']:.3f} / esp {met_05['specificity']:.3f})")
    met["segundos_entrenamiento"] = round(segundos, 1)
    met["parametros"] = int(modelo.count_params())
    met["canales"] = canales
    resultados.append(met); historias[ident] = hist
    curvas[titulo] = roc_curve(y_te, p)[:2]
    modelo.save(SALIDA / f"models/{ident}.keras")
    with open(SALIDA / f"metrics/{ident}.json", "w", encoding="utf-8") as fh:
        json.dump(met, fh, indent=2, ensure_ascii=False)
    print(f"ROC-AUC {met['roc_auc']:.4f} | sens {met['sensitivity']:.4f} | "
          f"esp {met['specificity']:.4f} | {segundos:.0f}s")
    del Xtr, Xva, Xte

tabla = pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)
tabla.to_csv(SALIDA / "metrics/comparacion.csv", index=False)
print("\n" + tabla[["modelo","roc_auc","sensitivity","specificity","f1","parametros",
                     "segundos_entrenamiento","ms_por_imagen"]].to_string(index=False))

## 5. Visualizaciones estáticas para el informe

In [ ]:
PAL = {"CNN propia (desde cero)": "#264653",
       "MobileNetV2 (ImageNet)": "#2a9d8f",
       "EfficientNetB0 (ImageNet)": "#e76f51"}

# --- ROC ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
for nombre, (fpr, tpr) in curvas.items():
    auc = tabla.loc[tabla.modelo.eq(nombre), "roc_auc"].iloc[0]
    ax.plot(fpr, tpr, lw=2.2, color=PAL[nombre], label=f"{nombre} — AUC {auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="0.6", lw=1, label="Azar — AUC 0.500")
ax.set(xlabel="1 − Especificidad (falsos positivos)", ylabel="Sensibilidad (verdaderos positivos)",
       title="Curvas ROC sobre el conjunto de prueba agrupado por paciente")
ax.legend(loc="lower right"); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(SALIDA / "figures/06_curvas_roc.png", dpi=200, bbox_inches="tight"); plt.close(fig)

# --- Matrices de confusion --------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, fila in zip(axes, resultados):
    cm = np.array(fila["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center", fontsize=14,
                    color="white" if cm[i, j] > cm.max() / 2 else "#1f2933", fontweight="bold")
    ax.set(xticks=[0,1], yticks=[0,1],
           xticklabels=["Pred. sano", "Pred. fractura"],
           yticklabels=["Real sano", "Real fractura"],
           title=f"{fila['modelo']}\nsens {fila['sensitivity']:.3f} · esp {fila['specificity']:.3f}")
fig.suptitle("Matrices de confusión (umbral 0.5)", fontweight="bold")
fig.tight_layout(); fig.savefig(SALIDA / "figures/06_matrices_confusion.png", dpi=200, bbox_inches="tight"); plt.close(fig)

# --- Comparacion metrica + coste --------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
met_cols = ["roc_auc", "sensitivity", "specificity", "f1"]
x = np.arange(len(met_cols)); ancho = 0.26
for k, fila in enumerate(resultados):
    axes[0].bar(x + k*ancho - ancho, [fila[c] for c in met_cols], ancho,
                label=fila["modelo"], color=PAL[fila["modelo"]])
axes[0].set(xticks=x, xticklabels=["ROC-AUC","Sensibilidad","Especificidad","F1"],
            ylim=(0,1.05), ylabel="Valor", title="Rendimiento por métrica")
axes[0].legend(fontsize=8); axes[0].grid(axis="y", alpha=.3)
for fila in resultados:
    axes[1].scatter(fila["segundos_entrenamiento"], fila["roc_auc"], s=220,
                    color=PAL[fila["modelo"]], label=fila["modelo"], zorder=3)
    axes[1].annotate(f"{fila['parametros']/1e6:.1f}M par.",
                     (fila["segundos_entrenamiento"], fila["roc_auc"]),
                     textcoords="offset points", xytext=(0, -22), ha="center", fontsize=9)
axes[1].set(xlabel="Tiempo de entrenamiento (s)", ylabel="ROC-AUC",
            title="Rendimiento frente a coste de entrenamiento")
axes[1].grid(alpha=.3); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(SALIDA / "figures/06_comparacion_modelos.png", dpi=200, bbox_inches="tight"); plt.close(fig)

# --- Curvas de aprendizaje --------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, (ident, h) in zip(axes, historias.items()):
    if "auc" in h: ax.plot(h["auc"], label="entrenamiento", color="#264653")
    if "val_auc" in h: ax.plot(h["val_auc"], label="validación", color="#e76f51")
    ax.set(title=ident, xlabel="Época", ylabel="AUC"); ax.legend(); ax.grid(alpha=.3)
fig.suptitle("Curvas de aprendizaje", fontweight="bold")
fig.tight_layout(); fig.savefig(SALIDA / "figures/06_curvas_aprendizaje.png", dpi=200, bbox_inches="tight"); plt.close(fig)

with open(SALIDA / "metrics/historias.json", "w", encoding="utf-8") as fh:
    json.dump(historias, fh, indent=2)
print("4 figuras guardadas")

## 6. `models.yaml` para la aplicación

In [ ]:
# Se genera desde los mismos valores usados al entrenar, para que la app no pueda
# quedar desincronizada del preprocesamiento real.
entradas = []
for (ident, titulo, _, _, canales), fila in zip(ESPECS, resultados):
    entradas.append(f"""  - id: {ident}
    name: "{titulo}"
    path: models/{ident}.keras
    framework: tensorflow
    format: keras
    input_size: [{LADO}, {LADO}]
    color_mode: {"rgb" if canales == 3 else "grayscale"}
    channels: {canales}
    normalization:
      type: none
    class_order: [sin_fractura, fractura]
    fracture_index: 1
    threshold: {fila['umbral']}
    output_type: probability
    metrics_path: metrics/{ident}.json
    last_conv_layer: null""")

yaml_txt = ("# Generado por notebooks/06-modelado.ipynb. No editar a mano.\n"
            "# El preprocesamiento aqui declarado es identico al del entrenamiento.\n"
            "models:\n" + "\n".join(entradas) + "\n")
(SALIDA / "models.yaml").write_text(yaml_txt, encoding="utf-8")
print(yaml_txt)

In [ ]:
mejor = tabla.iloc[0]
print("="*64); print("RESUMEN PARA EL INFORME"); print("="*64)
print(f"Cortes            : {len(inventario):,} ({int(inventario.etiqueta.sum()):,} pos / "
      f"{int((1-inventario.etiqueta).sum()):,} neg)")
print(f"Estudios          : {inventario.StudyInstanceUID.nunique():,}")
print(f"Particion         : {len(part_tr):,} / {len(part_va):,} / {len(part_te):,} cortes "
      f"({part_tr.StudyInstanceUID.nunique()}/{part_va.StudyInstanceUID.nunique()}/"
      f"{part_te.StudyInstanceUID.nunique()} estudios)")
print(f"Mejor modelo      : {mejor.modelo}")
print(f"  ROC-AUC         : {mejor.roc_auc:.4f}")
print(f"  Sensibilidad    : {mejor.sensitivity:.4f}")
print(f"  Especificidad   : {mejor.specificity:.4f}")
print(f"  F1              : {mejor.f1:.4f}")
print(f"  Inferencia      : {mejor.ms_por_imagen:.1f} ms/imagen")
print("="*64)